# Phase 0 — precision test + kill test (week 1)
Budget: ~1 session. Output of cell 2 is worth saving as a Kaggle Dataset (`fineweb-tok-200m`) so later runs can reuse it.

In [ ]:
# Settings: Accelerator = GPU T4 x2 (we use one), Internet = ON, Persistence = Files
REPO = "https://github.com/YOUR_USERNAME/loopthink"   # <- after you push this repo
!git clone -q $REPO /kaggle/working/loopthink || (cd /kaggle/working/loopthink && git pull -q)
%cd /kaggle/working/loopthink
!pip -q install tokenizers datasets huggingface_hub
import torch; print(torch.__version__, torch.cuda.get_device_name(0), "bf16:", torch.cuda.is_bf16_supported())

## 1. Data: tokenizer + 200M tokens (CPU-bound, ~30–60 min)

In [ ]:
!python scripts/prepare_pretrain_data.py --out /kaggle/working/tok200m --tokens 2e8

## 2. Precision test — 1,000 fp16 steps at r=16
PASS = no overflow, max activation ≪ 65504, loss scale doesn't collapse.

In [ ]:
!python scripts/precision_test.py --data_dir /kaggle/working/tok200m --out /kaggle/working/precision --steps 1000 --r 16
!cat /kaggle/working/precision/precision_verdict.json

## 3. Kill test — pretrain 200M tokens, fine-tune, check convergence signal
If the session dies, re-run this cell: pretraining resumes from its last 30-min checkpoint.

In [ ]:
!python scripts/kill_test.py --data_dir /kaggle/working/tok200m --out /kaggle/working/kill --tokens 2e8

In [ ]:
import json; print(json.dumps(json.load(open('/kaggle/working/kill/kill_test_verdict.json')), indent=2))
from IPython.display import Image, display
for s in ('official','heldout'): display(Image(f'/kaggle/working/kill/ft_{s}/eval/overview.png'))